# Traffic Sign Recognition & Road Safety Monitor
## Google Colab Model Training & Evaluation Pipeline

This interactive notebook executes the required workflow:
1. Clone the repository and install dependencies (`imagehash`, `torch`, `torchvision`, `scikit-learn`)
2. Acquire the GTSRB dataset
3. Perform perceptual hashing (`imagehash.phash`) and fast BK-Tree deduplication
4. Group images by track and perform reproducible 70% Train / 15% Val / 15% Test splitting (`seed=42`)
5. Train the Deep Convolutional Neural Network (`TrafficSignCNN`)
6. Evaluate model accuracy, CPU inference latency (target $\le 150$ ms), and safety class missed-sign rates (target $\le 5\%$)
7. Visualize confusion matrix and export to ONNX for Java `SafetyMonitor` deployment

### Step 1: Clone Repository & Install Dependencies

In [ ]:
# Clone the repository if running on fresh Colab VM
import os
if not os.path.exists('train/train.py'):
    !git clone https://github.com/shubhamjena28/traffic-sign-recognition-road-safety.git
    %cd traffic-sign-recognition-road-safety

!pip install -r train/requirements.txt


### Step 2: Verify GPU Acceleration

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active PyTorch Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Acceleration: {torch.cuda.get_device_name(0)}')


### Step 3: Dataset Acquisition & Structure
Download official GTSRB archives from ERDA mirror or use existing local structure.

In [ ]:
# Download official GTSRB training set (uncomment if downloading raw archive)
# !python data/download_datasets.py --dataset gtsrb --gtsrb-parts train


### Step 4: BK-Tree Perceptual Hash Deduplication & 70/15/15 Track Splitting
- Uses **BK-Tree metric indexing** to find duplicates without quadratic $O(N^2)$ comparisons
- Preserves intra-track video approach frames
- Eliminates cross-track duplicate leakage
- Splits tracks into 70% Train, 15% Val, 15% Test with fixed seed (42)

In [ ]:
!python train/prepare_gtsrb.py --data-dir data/raw/gtsrb/train --output-dir data/splits --threshold 4 --seed 42


### Step 5: Inspect Split Manifests & Track Independence

In [ ]:
import json
import pandas as pd

train_df = pd.read_csv('data/splits/train.csv')
val_df = pd.read_csv('data/splits/val.csv')
test_df = pd.read_csv('data/splits/test.csv')

with open('data/splits/split_summary.json') as f:
    summary = json.load(f)

print(f"Total Images: {summary['total_images']}")
print(f"Training:   {len(train_df):5d} images ({summary['train_percentage']}%) across {len(set(train_df['track_id']))} tracks")
print(f"Validation: {len(val_df):5d} images ({summary['val_percentage']}%) across {len(set(val_df['track_id']))} tracks")
print(f"Testing:    {len(test_df):5d} images ({summary['test_percentage']}%) across {len(set(test_df['track_id']))} tracks")

# Verify zero cross-track leakage
overlap = len(set(train_df['track_id']) & set(test_df['track_id']))
print(f"Cross-track leakage between train and test: {overlap} (Verified Isolated)")


### Step 6: Train Traffic Sign CNN Classifier

In [ ]:
!python train/train.py --train-csv data/splits/train.csv --val-csv data/splits/val.csv --epochs 25 --batch-size 64 --lr 0.001 --output-dir checkpoints


### Step 7: Evaluate Model against Road Safety Benchmarks

In [ ]:
!python train/evaluate.py --checkpoint checkpoints/best_model.pt --test-csv data/splits/test.csv --report-file data/splits/evaluation_report.txt --confusion-matrix data/splits/confusion_matrix.png


### Step 8: View Evaluation Results & Confusion Matrix

In [ ]:
from IPython.display import Image, display

with open('data/splits/evaluation_report.txt') as f:
    print(f.read())

if os.path.exists('data/splits/confusion_matrix.png'):
    display(Image('data/splits/confusion_matrix.png'))


### Step 9: Export Model to ONNX for Java Application Integration

In [ ]:
import sys
from pathlib import Path
sys.path.append('train')
from train import TrafficSignCNN

checkpoint = torch.load('checkpoints/best_model.pt', map_location='cpu')
model = TrafficSignCNN(num_classes=43)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

dummy_input = torch.randn(1, 3, 48, 48)
onnx_file = 'checkpoints/traffic_sign_classifier.onnx'
torch.onnx.export(
    model,
    dummy_input,
    onnx_file,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
)
print(f'[SUCCESS] Exported ONNX model to: {onnx_file}')
print(f'Model Size: {os.path.getsize(onnx_file) / (1024 * 1024):.2f} MB (Target <= 50 MB)')
